# BERT RAG with vectorDB
 Requirements:
 * Python 3x. (latest version| https://www.python.org/downloads/release/python-3145/)
 * pgvector database (https://github.com/pgvector/pgvector#installation)
 
 Notes: 
   - Using langchain to ingest and manage retrieval
   - BERT to convert text in to embeddings
   - PGVector to store embeddings
   - BERT


In [2]:
#install dependencies 
%pip install -U langchain langchain-postgres langchain-huggingface langchain-text-splitters sentence-transformers
%pip install -U pgvector psycopg2-binary transformers
%pip install -U langchain_community pypdf

  Using cached pgvector-0.3.6-py3-none-any.whl.metadata (13 kB)
Using cached pgvector-0.3.6-py3-none-any.whl (24 kB)
  Attempting uninstall: pgvector
    Found existing installation: pgvector 0.4.2
    Uninstalling pgvector-0.4.2:
      Successfully uninstalled pgvector-0.4.2

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached pgvector-0.4.2-py3-none-any.whl.metadata (19 kB)
Using cached pgvector-0.4.2-py3-none-any.whl (27 kB)
  Attempting uninstall: pgvector
    Found existing installation: pgvector 0.3.6
    Uninstalling pgvector-0.3.6:
      Successfully uninstalled pgvector-0.3.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-postgres 0.0.17 requires pgvector<0.4,>=0.2.5, but you have pgvector 0.4.2 which is

In [14]:
from langchain_postgres.vectorstores import PGVector
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

# initialize sentence transformer 
# all-MiniLM is a popular BERT based Sentence transformer.
# see https://huggingface.co/sentence-transformers
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# connect to pg database
host='localhost'
port='5432' # standard pg port
dbname='embeddingsdb'
user='postgres'
pwd='postgres'

connection = f"postgresql+psycopg://{user}:{pwd}@{host}:{port}/{dbname}"
collection_name = "knowledge_base"


# intialize and load documents to vector store
vector_store = PGVector(connection=connection, 
                        collection_name=collection_name, 
                        embeddings=embeddings,
                        use_jsonb=True
                        # set other params like embedding_lenght for efficiency
                        )




Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
# functions to vectorize and search.

def vectorize(text):
    """
     Vectorizes the given text and stores it in the vector DB.
    """
    if len(text) < 1 :
        print('Empty Text')
        return
    # initailize Text Splitter 
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,       # BERT optimal token window size
        chunk_overlap=50      # Context preservation between chunks
    )

    # Split text to chunks 
    docs = text_splitter.split_documents(text)
    return vector_store.add_documents(docs)

def search(query):
    print('===================================================')
    results = vector_store.similarity_search(query)
    for result in results:
        print(f" - {result.page_content}")
        print('---------------------------------------------')
    print('===================================================')
    

In [16]:

# Test with pdf document using langchain_community pdfloader
# CAUTION: run only once to avoid duplicates.
files = ["2021-bender-parrots.pdf","Elegoo Super Starter Kit for UNO V1.0.2019.09.17.pdf"]

for file in files:
    loader = PyPDFLoader(file) 
    document = loader.load()
    vectorize(document)


In [17]:
search("Power settings for adrino uno")

 - directly. If we tried to connect the motor straight to an UNO R3 board pin, there is a good chance 
that it could damage the UNO R3 board. So we use a power supply module provides power supply 
136 / 162
---------------------------------------------
 - power it with a 9V 1Amp power supply. 
The IR sensor is connected to the UNO directly since it uses almost no power. 
Component Required: 
(1) x Elegoo Uno R3
(1) x 830 tie-points breadboard
(1) x IR receiver module
(1) x IR remote
(1) x ULN2003 stepper motor driver module
(1) x Stepper motor
(1) x Power supply module
(1) x 9V1A Adapter
(9) x F-M wires (Female to Male DuPont wires)
(1) x M-M wire (Male to Male jumper wire)
159 / 162
---------------------------------------------
 - Relative humidity: 
Resolution: 16Bit 
Repeatability: ±1% RH 
Accuracy: At 25℃  ±5% RH 
Interchangeability: fully interchangeable 
Response time: 1 / e (63%) of 25℃ 6s 
 
1m / s air 6s 
Hysteresis: <± 0.3% RH 
Long-term stability: <± 0.5% RH / yr in 
Tempera

In [18]:
# Simple document search
search("documents by Tinit Gebru")




 - [66] Nurul Shamimi Kamaruddin, Amirrudin Kamsin, Lip Yee Por, and Hameedur
Rahman. 2018. A Review of Text Watermarking: Theory, Methods, and Applica-
tions. IEEE Access 6 (2018), 8011–8028. https://doi.org/10.1109/ACCESS.2018.
2796585
[67] Brendan Kennedy, Drew Kogon, Kris Coombs, Joseph Hoover, Christina Park,
Gwenyth Portillo-Wightman, Aida Mostafazadeh Davani, Mohammad Atari,
and Morteza Dehghani. 2018. A typology and coding manual for the study of
---------------------------------------------
 - Example picture 
145/162
---------------------------------------------
 - Linguistics, Hong Kong, China, 3615–3620. https://doi.org/10.18653/v1/D19-
1371
[13] Emily M. Bender and Batya Friedman. 2018. Data statements for natural lan-
guage processing: Toward mitigating system bias and enabling better science.
Transactions of the Association for Computational Linguistics 6 (2018), 587–604.
[14] Emily M. Bender and Alexander Koller. 2020. Climbing towards NLU: On
Meaning, Form, and Underst